## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 27 samples and 14875 columns.
First few rows of the data:


,time to last followup,survstatus,sample_title,diagnosis,tissue,Sex,age,Stage,cell type,differentiation,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,8.59,0,Lu1031A,Lung Adenocarcinoma,Lung,Female,57,IA,AD,Well,...,7.609257,6.404654,7.794737,7.763609,6.390111,6.155748,8.336788,7.752943,7.057440,7.787434
1,7.54,0,Lu1050A,Lung Adenocarcinoma,Lung,Male,58,IB,AD,Well,...,8.960444,7.783257,9.031700,9.397762,7.223517,7.342558,9.427546,8.544923,7.714072,9.010870
2,3.28,1,Lu1068A,Lung Adenocarcinoma,Lung,Female,57,IB,AD,Mod,...,8.411483,7.260510,8.586719,8.632693,6.731376,7.070203,9.103918,8.805666,8.002787,8.480207
3,11.96,0,Lu106A,Lung Adenocarcinoma,Lung,Male,63,IB,AD,Well,...,7.782319,6.535627,7.859483,8.173400,6.839753,6.052102,8.200441,7.069845,6.470527,7.914763
4,5.68,1,Lu113A,Lung Adenocarcinoma,Lung,Female,89,IA,AD,Well,...,8.120021,6.914451,8.065773,8.618442,6.605339,6.447653,8.449232,7.963453,7.131778,7.998035


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
